In [1]:
import subprocess
import os
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
from IPython.display import clear_output
from time import sleep
import seaborn as sns
import pickle
pd.set_option('display.float_format', lambda x: f'{x:.10f}')

In [58]:
class ips_omni_processor():
    def __init__(self, ips_path, omni_path, sun_spot_path):
        self.ips_path = ips_path
        self.omni_path = omni_path
        self.sun_spot_path = sun_spot_path
        
        self.file_name = [str((x - 1900)%100) if len(str((x - 1900)%100))>1 else '0'+str((x - 1900)%100) for x in range(1983, 2025) ]

        self.file_name = ['VLIST' + x for x in self.file_name]
        print("IPS files found in folders: \n", self.file_name)
        
        
        self.column_names = ['SOURCE', 'YRMNDY', 'UT', 'DIST', 'HLA',  'HLO', 'GLA', 'GLO', 'CARR', 
                'V', 'ER', 'SC-INDX', 'file']
        self.column_names = [x.lower() for x in self.column_names]
        # Here we concatenate files having read them one after the other as they ocuur in file_names.
        # Note: everytime the index needs to redifined to start from where the previous file left off.
        # An extra coloumn 'file' is added to keep track.
        self.df_test_1 = pd.DataFrame(columns=self.column_names)
        # print(f"{self.df_test_1.shape}")
        
        print(f"Making sun-spots data....")
        self.sun_spots = pd.read_csv(self.sun_spot_path, sep=";", names=['year', 'month', 'day', 'date_frac', 'day_total', 'day_std', 'num' ,'d_p'])
        self.sun_spots_recent = self.sun_spots.copy() 
        # Implementing a 'yrmndy' column from 'year', 'month' and 'day', and dropping the latter columns 
        self.sun_spots['yrmndy'] = pd.to_datetime(self.sun_spots[["year", "month", "day"]])
        self.sun_spots.drop(columns=['year', 'month', 'day', 'date_frac'], inplace=True)
        # Editing -1 in 'day_total' and 'day_std' to reflect np.nan
        self.sun_spots['day_total'] = self.sun_spots.day_total.apply(lambda x: np.nan if x == -1 else x)
        self.sun_spots['day_std'] = self.sun_spots.day_std.apply(lambda x: np.nan if x == -1 else x)
        
        print(f"Making omni data ........")
        self.omni = pd.read_csv(omni_path)
        self.omni.rename(columns={'Unnamed: 0':'yrmndy_hr'}, inplace=True)
        self.omni_df = self.omni[['yrmndy_hr', 'swSpeed_Smth_0']].copy()
        self.omni_start_date_str = str(self.omni_df.iloc[0,0])
        self.omni_start_date_prv_mnth_str = "-".join([
            x 
            if i != 1 
            else (str(int(x) - 1) if len(str(int(x) - 1)) == 2 else "0"+str(int(x) - 1)) 
            for i, x in enumerate(str(self.omni_df.iloc[0,0]).split("-"))
        ])
        
        print(f"Taking sun spots data from one month prior to omni start date....")
        self.sun_spots_recent = self.sun_spots.loc[self.sun_spots.yrmndy >= pd.to_datetime(self.omni_start_date_prv_mnth_str)].copy()
        
        print(f"Formatting IPS data .........")
        self.df_test_2 = self.make_ips_df().copy()
        # print(f"{self.df_test_2.shape}")
        print(f"Conditioning ips data .........")
        self.cond = (self.df_test_2.er_1==0) & (self.df_test_2.er < 50) & (self.df_test_2.dist < 0.8) & (self.df_test_2.dist > 0.25)
        self.df_test_2 = self.df_test_2.loc[self.cond]
        self.delta_t = (lambda x: x.days + x.seconds/(24*60*60))(pd.to_datetime('2050-01-01') - self.df_test_2.ut.min())
        self.df_test_2['time'] =  (1000 - (pd.to_datetime('2050-01-01') - self.df_test_2.ut).map(lambda x: 1000*(x.days + x.seconds/(24*60*60)))/self.delta_t)
        self.df_test_2 = pd.merge(left=self.df_test_2, right=self.sun_spots_recent[['yrmndy', 'day_total']], on='yrmndy', how='left')
        
        
        self.omni_start_date = round((lambda x: 1000 - 1000*(x.days + x.seconds/(24*60*60))/self.delta_t)(pd.to_datetime('2050-01-01') - 
                                                                                                     pd.to_datetime(self.omni_start_date_str)), 8)
        print(f"OMNI start date is calibrated to {self.omni_start_date}")
        
        self.delta_6 = 6*1e3/self.delta_t
        self.delta_8 = 8*1e3/self.delta_t
        self.df_test_2 = self.df_test_2.loc[self.df_test_2.time >= self.omni_start_date - self.delta_8] # IPS data set from 8days before start of omni data set
        self.drop_cols = ['source', 'yrmndy', 'ut', 'file', 'er_1'] ## columns to be dropped during training
        self.df_5 = self.df_test_2.drop(columns=self.drop_cols).copy()
        self.df_5.reset_index(inplace=True)
        self.df_5.rename(columns={'sc-indx': 'sc_indx'}, inplace=True)
        print(f"Columns to be scaled except 'time': {self.df_5.columns}")
        for column in self.df_5.columns:
            if 'time' not in column:
                self.df_5[column] = (self.df_5[column]- self.df_5[column].min())/ (self.df_5[column].max() - self.df_5[column].min())
        
        self.omni_df['yrmndy_hr'] = pd.to_datetime(self.omni_df.yrmndy_hr)
        self.omni_df['time'] = (1000 - (pd.to_datetime('2050-01-01') - self.omni_df.yrmndy_hr).map(lambda x: 1000*(x.days + x.seconds/(24*60*60)))/self.delta_t)
        self.omni_df.drop(columns=['yrmndy_hr'], inplace=True)
        

    def make_ips_df(self):
        for x in self.file_name:
            indx_strt = self.df_test_1.shape[0]
            df_test_0 = pd.read_csv(self.ips_path + x, sep=r'\s+', skipinitialspace=True, skiprows=8, header=None, 
                                names=self.column_names[0:-1], usecols=[y for y in range(len(self.column_names[0:-1]))])
            df_test_0.index = df_test_0.index + indx_strt
            df_test_0['file'] = [x for i in range(df_test_0.shape[0])]
            self.df_test_1 = pd.concat([self.df_test_1, df_test_0])  
        del df_test_0
        print(f"Shape of IPS data: {self.df_test_1.shape}")

        def v_err(row):
            if type(row.v) == str:
                if row.v[-4:] == '-999':
                    row.v = int(row.v[0:-4])
                    row['sc-indx'] = row.er
                    row.er = -999
                else:
                    row.v = int(row.v)
            return row

        # Implementing v_err 
        self.df_test_1 = self.df_test_1.apply(v_err, axis=1)
        # print(f"{self.df_test_1.shape}")

        # Implementing one-hot encoding for error 'er' values of -999 in new coloumn 'er_1'

        self.df_test_1['er_1'] = self.df_test_1.er.map(lambda x: 1 if x==-999 else 0)

        # The 'yrmndy' is read as an integer, so converting it into a string and adding required zeros
        # for the first few years in the 2000s
        def yr_mod(col):
            x = str(col)
            if len(str(x))<6:
                x = ''.join([ '0' for j in range(6 - len(x)) ]) + str(x) 
            else:
                 x 
            return x
        # Implementing datetime stamp on 'yrmndy'
        self.df_test_1['yrmndy'] = pd.to_datetime(self.df_test_1.yrmndy.map(yr_mod), format='%y%m%d')
        print(f"{self.df_test_1.shape}")

        # Implementing conversion to datetime by adding yrmndy to it
        self.df_test_1['ut'] = self.df_test_1.yrmndy + pd.to_timedelta(self.df_test_1.ut, unit='h')
        return self.df_test_1

    def find_ranked_er(self, time, time_delta):
        """
        Parameters:
        --------------------------------------------------
        time: time in df_5.time format
        time_delta: time interval in df_5.time format


        Returns:
        -------------------------------------------------
        np.array of ranked list of df_5 indices according to least error i.e. df_5.er
        """
        df = self.df_5.loc[(self.df_5.time <= time) & (self.df_5.time > time - time_delta)]
        if len(df) > 0:
            return df.er.sort_values().index
        else:
            return np.array([])

    def fill_bracket(self, time_0, time_1, intervals):
        """
        Params:
        -----------------------------------------------
        time_0: time in time formart of df_5
        time_1: < time_0
        intervals: total # equally spaced time intervals b/w time_0 and time_1

        Returns:
        -------------------------------------------------
        'intervals' many obs- with one obs of least error in each interval, as a pd.DataFrame().values.
        If no value is found in an interval, then its filled with the remainder set once all the intervals have been filled with the best within them 
        """
        time_delta = (time_0 - time_1)/intervals # size of each interval
        # print(time_delta)
        df = self.df_5.loc[(self.df_5.time <= time_0) & (self.df_5.time > time_1)].copy()


        if len(df) <= intervals:
            bracket = df
            # print('less')
        else:
            # print('more')
            # First fill in the intervals with the best obs from the same interval
            rest_obs_id = np.array([])
            empty_list = [] # list for intervals with no obs
            bracket = []   # to store obs rows
            for i in range(intervals):
                obs_id = self.find_ranked_er(time_0 - i*time_delta, time_delta)     ## get list of indices with obs in the interval ranked acc. to error i.e. df_5.er
                # print(i)
                if len(obs_id) != 0:
                    bracket.append(df.loc[df.index==obs_id[0]].values.reshape((1,-1)).tolist())
                    if len(obs_id) > 1:
                        obs_id = np.delete(obs_id, 0)
                        rest_obs_id = np.concatenate((rest_obs_id, obs_id))   # storing the ranked obs indices for filling unfilled intervals
                    # break
                else:
                    empty_list.append(i) # keeping track of empty intervals

            bracket = np.array(bracket)
            bracket = bracket.reshape((-1,len(self.df_5.columns))) 
            # print(bracket.shape)

            # Fill the rest of the intervals if any with the remainder of obs from other intervals
            rest_obs_id = rest_obs_id.astype(int)
            if len(rest_obs_id) > 0:
                for i, obs_j in zip(empty_list, rest_obs_id):
                    # print(obs_j in list(df.index))
                    bracket = np.concatenate((bracket, df.loc[df.index==obs_j].values.reshape((1,-1))))

        # Arranging the dataset according to time
        bracket = pd.DataFrame(bracket, columns=df.columns)
        bracket.drop(columns=['index'], inplace=True)
        bracket.sort_values('time', ascending=False, inplace=True)
        bracket.reset_index(inplace=True)
        bracket.drop(columns='index', inplace=True)
        del df
        return bracket

    def make_training_data(self, ips_data, omni_data, min_input_len=20):
        """
        Construct training data as follows:
        Begining at every hour i of the omni_data construct
        the target y: 16 omni_data obs into the future begining at hour i i.e. omin_data 4 days into the future
        the input  x: best ips_data from 8-days into the past begining from the hour i as a pd.DataFrame().values


        The length of the omni_data controls the length of the output.
        If the input x generated doesn't have length of min_len_input then the data corresponding to the hour i is not considered.

        Params:
        -----------------------------------------------------------------------------------------------------------------------------------------------------------
        omni_data: omni data as pd.DataFrame with time as in time in df_5 and smoothed hourly solar wind speed.
        ips_data: ips pd.DataFrame() with relevant columns and time as above along with sun spot numbers for the relevant days.
        min_input_len: min number of ips_data in the past 8-days- if less, then the data point is skipped.

        Returns:
        -----------------------------------------------------------------------------------------------------------------------------------------------------------
        A list s.t. each row is a list [i, x, y] where 
        i: is the index starting from 0
        y: is the above target and 
        x: is the above (2dim with x.shape:(32, #selected columns from ips_data)) input

        A missing list containing rows [i, missed] where
        i: the index where no.of x data generated is not of length of 32
        missed: length of x data
        """

        out_data = [] 
        j = 0  # index 
        k = 0  # no.of samples skipped
        missing = []
        for i in range(len(omni_data) - 16):
            time = omni_data.iloc[i].time
            x_brckt = self.fill_bracket(time, time - self.delta_8, 32) # x_brckt has max len 32, it can be smaller
            x_brckt_len = len(x_brckt)
            # Do not make sample if x_brckt has len < 20
            if x_brckt_len < min_input_len:
                k = k + 1
                continue

            # adding an extra column in x for keeping track of the time of the input
            # this column has time as its entry for the first len(x_brckt) entries  
            # and then np.zeros for the remainding entries upto 32 if len(x_brckt) < 32
            # print(len(x_brckt))


            if x_brckt_len < 32:
                x_brckt_0 = pd.DataFrame(np.zeros(11*(32 - x_brckt_len)).reshape((32 - x_brckt_len), -1), columns=x_brckt.columns)
                x_brckt = pd.concat([x_brckt, x_brckt_0])
            if x_brckt_len == 32:
                time_0 = time*np.ones(32)
            else:
                time_0 = np.concatenate([time*np.ones(x_brckt_len), np.zeros(32 - x_brckt_len)])
                # print(j)
            # time_0 = time*np.ones(32)
            # x_brckt['time_trgt'] = pd.Series(time_0)
            x_brckt['time_trgt'] = time_0
            # Adding input column to indicate missing rows as 0
            # x_brckt['input'] = pd.Series(np.concatenate([np.ones(x_brckt_len), np.zeros(32 - x_brckt_len)]), dtype=float)
            x_brckt['input'] = np.concatenate([np.ones(x_brckt_len), np.zeros(32 - x_brckt_len)])
            x_brckt['time'] = x_brckt['time_trgt'] - x_brckt['time']


            # Test x_brckt['input'].sum()
            # print(x_brckt['input'].values.sum(), x_brckt_len, pd.Series(np.concatenate([np.ones(x_brckt_len), np.zeros(32 - x_brckt_len)]), dtype=float).values.sum())
            if x_brckt_len != x_brckt['input'].sum():
                print(x_brckt['input'].sum(), x_brckt_len, x_brckt.input.values)


            # Uncomment line below to return a list with X and y as pd.DataFrames
            # out_data.append([j, x_brckt, omni_data.iloc[i: i + 16] ]) 

            out_data.append([j] + list(x_brckt.values.reshape(-1)) + list(omni_data.iloc[i: i+16, 0].values.reshape(-1))) # choosing only one column from omni data

            # Keep track of x_brckt when len < 32
            if x_brckt_len < 32:
                missing.append([j, x_brckt_len])

            j = j + 1

        print(f"{k} Data points skipped due to lack of atleast {min_input_len} IPS data points in the past 8-days.")

        missing_df = pd.DataFrame(missing, columns=['id', 'missed'])
        if len(missing) > 0:
            print(missing_df.describe().to_string())
        print(f"{j} Data points made.")
        # return out_data, missing

        clmns_ips = list(ips_data.columns)
        clmns_ips.pop(0)
        clmns_ips.append('time_trgt')
        clmns_ips.append('input')
        print(clmns_ips, len(clmns_ips))
        clmns_input = []
        for i in range(32):
            for clmn in clmns_ips:
                clmns_input.append(f"X_{clmn}_{i}")
        # print(clmns_input)
        clmns_omni = list(omni_data.columns)
        clmns_target = []
        for i in range(16):
            for clmn in clmns_omni:
                if 'time' not in clmn:  # choosing only one column from omni data
                    clmns_target.append(f"y_{clmn}_{i}")
        clmns_data = ['idx'] + clmns_input + clmns_target

        out_df = pd.DataFrame(out_data, columns=clmns_data)

        return out_df, missing_df

    def make_final_data(self, ips_df, omni_df):
        # Making the final data 
        out_df, missing_df = self.make_training_data(ips_df, omni_df)
        
        print(f"scaling output data's time columns")
        for column in out_df.columns:
            if "time" in column:
                out_df[column] = out_df[column]/1000
        return out_df, missing_df
        

In [7]:
ips_omni = ips_omni_processor(ips_path="../data/test_dwnld/", sun_spot_path='../data/SN_d_tot_V2.0.csv', omni_path='../data/omni_avg_normalised_smoothed_extened.csv')

IPS files found in folders: 
 ['VLIST83', 'VLIST84', 'VLIST85', 'VLIST86', 'VLIST87', 'VLIST88', 'VLIST89', 'VLIST90', 'VLIST91', 'VLIST92', 'VLIST93', 'VLIST94', 'VLIST95', 'VLIST96', 'VLIST97', 'VLIST98', 'VLIST99', 'VLIST00', 'VLIST01', 'VLIST02', 'VLIST03', 'VLIST04', 'VLIST05', 'VLIST06', 'VLIST07', 'VLIST08', 'VLIST09', 'VLIST10', 'VLIST11', 'VLIST12', 'VLIST13', 'VLIST14', 'VLIST15', 'VLIST16', 'VLIST17', 'VLIST18', 'VLIST19', 'VLIST20', 'VLIST21', 'VLIST22', 'VLIST23', 'VLIST24']
Making sun-spots data....
Making omni data ........
Taking sun spots data from one month prior to omni start date....
Formatting IPS data .........


/tmp/ipykernel_7016/3448239894.py:86: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.df_test_1 = pd.concat([self.df_test_1, df_test_0])


Shape of IPS data: (132077, 13)
(132077, 14)
Conditioning ips data .........
OMNI start date is calibrated to 200.02282467
Columns to be scaled except 'time': Index(['index', 'dist', 'hla', 'hlo', 'gla', 'glo', 'carr', 'v', 'er',
       'sc_indx', 'time', 'day_total'],
      dtype='object')


In [8]:
full_1_df, missing = ips_omni.make_final_data(ips_omni.df_5, ips_omni.omni_df.iloc[int(ips_omni.omni_df.shape[0]*0.5): int(ips_omni.omni_df.shape[0]*0.5)+ 100])

0 Data points skipped due to lack of atleast 20 IPS data points in the past 8-days.
                 id        missed
count 28.0000000000 28.0000000000
mean  51.5714285714 30.6428571429
std   15.3729646564  0.7800420556
min   34.0000000000 28.0000000000
25%   40.7500000000 31.0000000000
50%   47.5000000000 31.0000000000
75%   55.2500000000 31.0000000000
max   83.0000000000 31.0000000000
84 Data points made.
['dist', 'hla', 'hlo', 'gla', 'glo', 'carr', 'v', 'er', 'sc_indx', 'time', 'day_total', 'time_trgt', 'input'] 13
caling output data's time columns


In [18]:
int(ips_omni.omni_df.shape[0]*0.25)

57951

In [21]:
full_1_df.y_swSpeed_Smth_0_6

0    0.3533854167
1    0.3558333333
2    0.3584895833
3    0.3618229167
4    0.3655208333
         ...     
79   0.4963020833
80   0.4991666667
81   0.5020312500
82   0.5051041667
83   0.5080729167
Name: y_swSpeed_Smth_0_6, Length: 84, dtype: float64

In [22]:
ips_omni.omni_df

,swSpeed_Smth_0,time
0,0.6710416667,200.0228246700
1,0.6711538462,200.0245331000
2,0.6693750000,200.0262415300
3,0.6640000000,200.0279499600
4,0.6608593750,200.0296584000
...,...,...
231799,0.4460937500,596.0358887100
231800,0.4477083333,596.0375971400
231801,0.4494791667,596.0393055800
231802,0.4515625000,596.0410140100


In [11]:
full_1_df.X_time_0

0    0.0000261732
1    0.0000278816
2    0.0000295901
3    0.0000312985
4    0.0000330069
         ...     
79   0.0000021014
80   0.0000038098
81   0.0000016743
82   0.0000016572
83   0.0000033656
Name: X_time_0, Length: 84, dtype: float64

In [12]:
full_1_df.X_v_0

0    0.3257491676
1    0.3257491676
2    0.3257491676
3    0.3257491676
4    0.3257491676
         ...     
79   0.1298557159
80   0.1298557159
81   0.2430632630
82   0.3218645949
83   0.3218645949
Name: X_v_0, Length: 84, dtype: float64

In [23]:
ips_omni.omni_df.iloc[:,0]

0        0.6710416667
1        0.6711538462
2        0.6693750000
3        0.6640000000
4        0.6608593750
             ...     
231799   0.4460937500
231800   0.4477083333
231801   0.4494791667
231802   0.4515625000
231803   0.4539583333
Name: swSpeed_Smth_0, Length: 231804, dtype: float64

## Downloaded data

In [4]:
test_df = pd.read_csv("data/data_generated/test/test_ips_omni_df")

In [5]:
test_df.columns

Index(['idx', 'X_dist_0', 'X_hla_0', 'X_hlo_0', 'X_gla_0', 'X_glo_0',
       'X_carr_0', 'X_v_0', 'X_er_0', 'X_sc_indx_0',
       ...
       'y_swSpeed_Smth_0_11', 'y_time_11', 'y_swSpeed_Smth_0_12', 'y_time_12',
       'y_swSpeed_Smth_0_13', 'y_time_13', 'y_swSpeed_Smth_0_14', 'y_time_14',
       'y_swSpeed_Smth_0_15', 'y_time_15'],
      dtype='object', length=449)

In [5]:
os.getcwd()

'/home/koba/Documents/ML/solar_weather/solar_wind/notebooks'

In [54]:
import sys
import os

# print(os.getcwd())
# os.chdir("..")
# print(os.getcwd())
sys.path.insert(0, "..")
# # Add the parent directory to sys.path
# # sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))

from codes.make_dataset import DatasetHist

In [55]:
DatasetHist("../data/data_generated/train/train_ips_omni_df.csv")
DatasetHist("../data/data_generated/val/val_ips_omni_df.csv")
DatasetHist("../data/data_generated/test/test_ips_omni_df.csv")

Input features are ['dist', 'hla', 'hlo', 'gla', 'glo', 'carr', 'v', 'er', 'sc_indx', 'time', 'day_total', 'time_trgt', 'input']
Target features are ['swSpeed_Smth_0']
begins: 0 ends: 71991
Input features are ['dist', 'hla', 'hlo', 'gla', 'glo', 'carr', 'v', 'er', 'sc_indx', 'time', 'day_total', 'time_trgt', 'input']
Target features are ['swSpeed_Smth_0']
begins: 71994 ends: 77399
Input features are ['dist', 'hla', 'hlo', 'gla', 'glo', 'carr', 'v', 'er', 'sc_indx', 'time', 'day_total', 'time_trgt', 'input']
Target features are ['swSpeed_Smth_0']
begins: 77456 ends: 81211


In [66]:
full_df = pd.read_csv("../data/data_generated/full_df.csv")

In [67]:
for column in full_df.columns:
    if full_df[column].isnull().any():
        print(column, full_df[column].isnull().sum()) # all NaNs comming from sunspot data

In [59]:
ips_omni = ips_omni_processor(ips_path="../data/test_dwnld/", sun_spot_path='../data/SN_d_tot_V2.0.csv', omni_path='../data/omni_avg_normalised_smoothed_extened.csv')

IPS files found in folders: 
 ['VLIST83', 'VLIST84', 'VLIST85', 'VLIST86', 'VLIST87', 'VLIST88', 'VLIST89', 'VLIST90', 'VLIST91', 'VLIST92', 'VLIST93', 'VLIST94', 'VLIST95', 'VLIST96', 'VLIST97', 'VLIST98', 'VLIST99', 'VLIST00', 'VLIST01', 'VLIST02', 'VLIST03', 'VLIST04', 'VLIST05', 'VLIST06', 'VLIST07', 'VLIST08', 'VLIST09', 'VLIST10', 'VLIST11', 'VLIST12', 'VLIST13', 'VLIST14', 'VLIST15', 'VLIST16', 'VLIST17', 'VLIST18', 'VLIST19', 'VLIST20', 'VLIST21', 'VLIST22', 'VLIST23', 'VLIST24']
Making sun-spots data....
Making omni data ........
Taking sun spots data from one month prior to omni start date....
Formatting IPS data .........


/tmp/ipykernel_3737/4119590109.py:86: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.df_test_1 = pd.concat([self.df_test_1, df_test_0])


Shape of IPS data: (132077, 13)
(132077, 14)
Conditioning ips data .........
OMNI start date is calibrated to 200.02282467
Columns to be scaled except 'time': Index(['index', 'dist', 'hla', 'hlo', 'gla', 'glo', 'carr', 'v', 'er',
       'sc_indx', 'time', 'day_total'],
      dtype='object')


In [60]:
ips_omni.sun_spots[['yrmndy', 'day_total']].isnull().sum()

yrmndy          0
day_total    3247
dtype: int64

In [61]:
ips_omni.df_test_2.day_total.isnull().sum()

np.int64(0)

In [63]:
ips_omni.df_test_2.loc[ips_omni.df_test_2.day_total.isnull()].yrmndy

Series([], Name: yrmndy, dtype: datetime64[ns])

In [64]:
ips_omni.sun_spots_recent.yrmndy.min()

Timestamp('1996-07-02 00:00:00')

In [50]:
omni_df = pd.read_csv('../data/omni_avg_normalised_smoothed_extened.csv')
omni_df.rename(columns={'Unnamed: 0':'yrmndy_hr'}, inplace=True)
omni_df = omni_df[['yrmndy_hr', 'swSpeed_Smth_0']]

In [53]:
omni_df.yrmndy_hr.min()

'1996-08-01 12:00:00'

In [65]:
print(ips_omni.omni_start_date_str, ips_omni.omni_start_date_prv_mnth_str)

1996-08-01 12:00:00 1996-07-01 12:00:00


In [37]:
ips_omni.sun_spots.yrmndy.isin(ips_omni.df_test_2.loc[ips_omni.df_test_2.day_total.isnull()].yrmndy.values).sum()

np.int64(40)

In [7]:
[x[2:-2] for x in list(full_df.columns)[1: 13 + 1]]

['dist',
 'hla',
 'hlo',
 'gla',
 'glo',
 'carr',
 'v',
 'er',
 'sc_indx',
 'time',
 'day_total',
 'time_trgt',
 'input']

In [13]:
[x[2:-2] for x in list(full_df.columns)[1 + 13*32: 1 + 13*32 + 1]]

['swSpeed_Smth_0']

In [68]:
full_df.X_input_31.describe()

count   81212.0000000000
mean        0.7947593952
std         0.4038798182
min         0.0000000000
25%         1.0000000000
50%         1.0000000000
75%         1.0000000000
max         1.0000000000
Name: X_input_31, dtype: float64

In [40]:
(full_df.X_time_trgt_0 == full_df.X_time_trgt_21).describe()

count     81212
unique        2
top        True
freq      77560
dtype: object

## Making train val and test splits

In [69]:
# Required so that there is no overlap between train, val and test. The minimum interval is 4 days as the target spans 4 days into the future.
fourdays = ips_omni.delta_8/2000 

In [70]:
# Training data upto 2013
((2050 - ips_omni.delta_t*(1 - full_df.X_time_trgt_0 )//365) <= 2013).sum()

np.int64(71992)

In [71]:
# validation data after 2013 and before 2017
(((2050 - ips_omni.delta_t*(1 - full_df.X_time_trgt_0 )//365) > 2013) & ((2050 - ips_omni.delta_t*(1 - full_df.X_time_trgt_0 )//365) < 2017)).sum()

np.int64(5408)

In [72]:
# test data from 2017
((2050 - ips_omni.delta_t*(1 - full_df.X_time_trgt_0 )//365) >= 2017).sum()

np.int64(3812)

In [73]:
full_df.index[10]

10

In [74]:
# Finding the index for marking the train val slpit without overlap
train_val_mark_0 = int(((2050 - ips_omni.delta_t*(1 - full_df.X_time_trgt_0 )//365) <= 2013).sum())
print(train_val_mark_0)
train_val_mark_1 = full_df.loc[(full_df.index >= train_val_mark_0) & (full_df.X_time_trgt_0 > full_df.iloc[train_val_mark_0].X_time_trgt_0 + fourdays)].index[0]
print(train_val_mark_1)

71992
71994


In [75]:
((2050 - ips_omni.delta_t*(1 - full_df.X_time_trgt_0 )//365) <= 2013)[train_val_mark_0 -1] 

np.True_

In [77]:
# Finding the index for marking the val test slpit without overlap
val_test_mark_0 = ((2050 - ips_omni.delta_t*(1 - full_df.X_time_trgt_0 )//365) < 2017).sum()
print(val_test_mark_0)
val_test_mark_1 = full_df.loc[(full_df.index >= val_test_mark_0) & (full_df.X_time_trgt_0 > full_df.iloc[val_test_mark_0].X_time_trgt_0 + fourdays)].index[0]
print(val_test_mark_1)

77400
77456


In [78]:
(val_test_mark_1 - full_df.shape[0])/full_df.shape[0]

np.float64(-0.046249322760183224)

In [79]:
train_ips_omni_df = full_df.iloc[:train_val_mark_0]
val_ips_omni_df = full_df.iloc[train_val_mark_1:val_test_mark_0]
test_ips_omni_df = full_df.iloc[val_test_mark_1:]

In [80]:
train_ips_omni_df.to_csv("../data/data_generated/train/train_ips_omni_df.csv", index=False)
val_ips_omni_df.to_csv("../data/data_generated/val/val_ips_omni_df.csv", index=False)
test_ips_omni_df.to_csv("../data/data_generated/test/test_ips_omni_df.csv", index=False)

In [26]:
full_df.loc[(full_df.index >= train_val_mark) & (full_df.X_time_trgt_0 > full_df.iloc[train_val_mark].X_time_trgt_0 + fourdays)]

,idx,X_dist_0,X_hla_0,X_hlo_0,X_gla_0,X_glo_0,X_carr_0,X_v_0,X_er_0,X_sc_indx_0,...,y_swSpeed_Smth_0_11,y_time_11,y_swSpeed_Smth_0_12,y_time_12,y_swSpeed_Smth_0_13,y_time_13,y_swSpeed_Smth_0_14,y_time_14,y_swSpeed_Smth_0_15,y_time_15
71994,71994,0.9433962264,0.6510067114,0.2715231788,0.5987261146,0.2944444444,0.5921052632,0.1770255272,0.5918367347,0.0005505376,...,0.4627083333,0.4509660078,0.4621354167,0.4509677162,0.4613541667,0.4509694247,0.4602604167,0.4509711331,0.4591145833,0.4509728416
71995,71995,0.9433962264,0.6510067114,0.2715231788,0.5987261146,0.2944444444,0.5921052632,0.1770255272,0.5918367347,0.0005505376,...,0.4621354167,0.4509677162,0.4613541667,0.4509694247,0.4602604167,0.4509711331,0.4591145833,0.4509728416,0.4581250000,0.4509745500
71996,71996,0.9433962264,0.6510067114,0.2715231788,0.5987261146,0.2944444444,0.5921052632,0.1770255272,0.5918367347,0.0005505376,...,0.4613541667,0.4509694247,0.4602604167,0.4509711331,0.4591145833,0.4509728416,0.4581250000,0.4509745500,0.4570312500,0.4509762584
71997,71997,0.9433962264,0.6510067114,0.2715231788,0.5987261146,0.2944444444,0.5921052632,0.1770255272,0.5918367347,0.0005505376,...,0.4602604167,0.4509711331,0.4591145833,0.4509728416,0.4581250000,0.4509745500,0.4570312500,0.4509762584,0.4564062500,0.4509779668
71998,71998,0.9433962264,0.6510067114,0.2715231788,0.5987261146,0.2944444444,0.5921052632,0.1770255272,0.5918367347,0.0005505376,...,0.4591145833,0.4509728416,0.4581250000,0.4509745500,0.4570312500,0.4509762584,0.4564062500,0.4509779668,0.4554687500,0.4509796753
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81207,81207,0.6792452830,0.7449664430,0.7748344371,0.7770700637,0.2805555556,0.9157894737,0.2408435072,0.1224489796,0.0005182796,...,0.4945833333,0.5888006749,0.4920833333,0.5888023833,0.4893229167,0.5888040918,0.4870312500,0.5888058002,0.4835416667,0.5888075086
81208,81208,0.6792452830,0.7449664430,0.7748344371,0.7770700637,0.2805555556,0.9157894737,0.2408435072,0.1224489796,0.0005182796,...,0.4920833333,0.5888023833,0.4893229167,0.5888040918,0.4870312500,0.5888058002,0.4835416667,0.5888075086,0.4810416667,0.5888092171
81209,81209,0.6792452830,0.7449664430,0.7748344371,0.7770700637,0.2805555556,0.9157894737,0.2408435072,0.1224489796,0.0005182796,...,0.4893229167,0.5888040918,0.4870312500,0.5888058002,0.4835416667,0.5888075086,0.4810416667,0.5888092171,0.4781250000,0.5888109255
81210,81210,0.6792452830,0.7449664430,0.7748344371,0.7770700637,0.2805555556,0.9157894737,0.2408435072,0.1224489796,0.0005182796,...,0.4870312500,0.5888058002,0.4835416667,0.5888075086,0.4810416667,0.5888092171,0.4781250000,0.5888109255,0.4752083333,0.5888126339


In [4]:
val_df = pd.read_csv("../data/data_generated/val/val_ips_omni_df.csv")

In [9]:
val_df.loc[val_df.idx==val_df.idx[0]]

,idx,X_dist_0,X_hla_0,X_hlo_0,X_gla_0,X_glo_0,X_carr_0,X_v_0,X_er_0,X_sc_indx_0,...,y_swSpeed_Smth_0_11,y_time_11,y_swSpeed_Smth_0_12,y_time_12,y_swSpeed_Smth_0_13,y_time_13,y_swSpeed_Smth_0_14,y_time_14,y_swSpeed_Smth_0_15,y_time_15
0,71994,0.9433962264,0.6510067114,0.2715231788,0.5987261146,0.2944444444,0.5921052632,0.1770255272,0.5918367347,0.0005505376,...,0.4627083333,0.4509660078,0.4621354167,0.4509677162,0.4613541667,0.4509694247,0.4602604167,0.4509711331,0.4591145833,0.4509728416


In [11]:
idx_values = val_df.idx.values

In [15]:
val_df.loc[val_df.idx==idx_values[0]].values

array([[7.19940000e+04, 9.43396226e-01, 6.51006711e-01, 2.71523179e-01,
        5.98726115e-01, 2.94444444e-01, 5.92105263e-01, 1.77025527e-01,
        5.91836735e-01, 5.50537634e-04, 4.27110535e-07, 3.45609065e-01,
        4.50947215e-01, 1.00000000e+00, 5.84905660e-01, 6.91275168e-01,
        1.72185430e-01, 6.36942675e-01, 2.58333333e-01, 5.92105263e-01,
        1.86459489e-01, 4.69387755e-01, 2.14623656e-04, 2.40889283e-06,
        3.45609065e-01, 4.50947215e-01, 1.00000000e+00, 2.45283019e-01,
        9.12751678e-01, 7.48344371e-01, 8.91719745e-01, 4.66666667e-01,
        5.92105263e-01, 1.58157603e-01, 1.22448980e-01, 1.70752688e-03,
        7.38043290e-06, 3.45609065e-01, 4.50947215e-01, 1.00000000e+00,
        9.62264151e-01, 6.44295302e-01, 2.71523179e-01, 5.98726115e-01,
        3.33333333e-01, 5.92105263e-01, 1.99223085e-01, 9.18367347e-01,
        5.82795699e-04, 4.13099125e-05, 2.77620397e-01, 4.50947215e-01,
        1.00000000e+00, 6.03773585e-01, 6.84563758e-01, 1.721854